In [113]:
import os
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video
import imageio
from statistics import mean
import robomimic.utils.file_utils as FileUtils
from PIL import Image, ImageDraw, ImageFont
import zarr
from diffusion_policy.common.replay_buffer import ReplayBuffer
from filelock import FileLock
from diffusion_policy.codecs.imagecodecs_numcodecs import register_codecs, Jpeg2k
import pdb
from tqdm import tqdm

In [114]:
og_redcube_data = h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/robomimic/datasets/lift/ph/image_abs.hdf5', 'r')
print(og_redcube_data['data']['demo_9'].keys())

<KeysViewHDF5 ['actions', 'dones', 'next_obs', 'obs', 'rewards', 'states']>


In [107]:
trial_hammer_basepath = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/alift_mugred_31_14_13_52/'
obsdict_agentview = np.load(trial_hammer_basepath + 'obsdict_agentview.npy', 'r')
obsdict_eyeinhand = np.load(trial_hammer_basepath + 'obsdict_eyeinhand.npy', 'r')
obsdict_robot0s = np.load(trial_hammer_basepath + 'obsdict_robot0s.npy', 'r')
rewards = np.load(trial_hammer_basepath + 'rewards.npy', 'r')
states = np.load(trial_hammer_basepath + 'startstates.npy', 'r')
actions = np.load(trial_hammer_basepath + 'actions.npy', 'r')

In [108]:
print('hi', trial_hammer_basepath)
obsdict_agentview = (obsdict_agentview.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_agentview', obsdict_agentview.shape)

obsdict_eyeinhand = (obsdict_eyeinhand.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_eyeinhand', obsdict_eyeinhand.shape)

obsdict_robot0s = obsdict_robot0s.transpose(0,2,1,3).reshape(13*8,1008,9)[:-4]
print('obsdict_robot0s', obsdict_robot0s.shape)

actions = np.load(trial_hammer_basepath + 'actions.npy', 'r')
print(actions.shape)
actions = actions[:,:,:8,:].transpose(0,2,1,3).reshape(13*8,1008,7)[:-4]
print('actions', actions.shape)

rewards = rewards.transpose(1,0)
print('rewards', rewards.shape)

states = np.repeat(np.array(states)[np.newaxis, :, :], obsdict_agentview.shape[0], axis=0)
print('states', states.shape)

hi /proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/alift_mugred_31_14_13_52/
obsdict_agentview (100, 1008, 84, 84, 3)
obsdict_eyeinhand (100, 1008, 84, 84, 3)
obsdict_robot0s (100, 1008, 9)
(13, 1008, 8, 7)
actions (100, 1008, 7)
rewards (100, 1008)
states (100, 1008, 32)


expected:

hi
obsdict_agentview (100, 1008, 84, 84, 3)
obsdict_eyeinhand (100, 1008, 84, 84, 3)
obsdict_robot0s (100, 1008, 9)
(13, 1008, 8, 7)
actions (100, 1008, 7)
rewards (100, 1008)
states (100, 1008, 32)

In [112]:
''' CREATE 1 DATASET '''
new_data = {'data':{}}

count = 0
for datapt in range(0,rewards.shape[1]):
    success = np.max(rewards[:,datapt])
    if success:
        stop_demo = min(np.argwhere(rewards[:,datapt]==1))[0]+16
    else:
        stop_demo = 100
    new_data['data'][f'demo_{count}'] = {
        'rewards': rewards[:stop_demo,datapt],
        'success': [success],
        'og_pt': [datapt],
        'states': states[:stop_demo,datapt,:],
        'actions': actions[:stop_demo, datapt],
        'obs': {
            'agentview_image': obsdict_agentview[:stop_demo, datapt],
            'robot0_eye_in_hand_image': obsdict_eyeinhand[:stop_demo, datapt],
            'robot0_eef_pos': obsdict_robot0s[:stop_demo, datapt, :3],
            'robot0_eef_quat': obsdict_robot0s[:stop_demo, datapt, 3:7],
            'robot0_gripper_qpos': obsdict_robot0s[:stop_demo, datapt, 7:],
        }
    }
    count+=1    

      
# Open HDF5 file and write in the data_dict structure and info
savepath = trial_hammer_basepath+'data_all.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')


datagrp.attrs['env_args'] = og_redcube_data['data'].attrs['env_args']
datagrp.attrs['total'] = len(new_data['data'])


for demo in new_data['data']:
    demogrp = datagrp.create_group(demo)
    
    actionsdset = demogrp.create_dataset('actions', data = new_data['data'][demo]['actions'])
    rewardsdset = demogrp.create_dataset('rewards', data = new_data['data'][demo]['rewards'])
    successdset = demogrp.create_dataset('success', data = new_data['data'][demo]['success'])
    statesdset = demogrp.create_dataset('states', data = new_data['data'][demo]['states'])
    statesdset = demogrp.create_dataset('og_pt', data = new_data['data'][demo]['og_pt'])

    obsgrp = demogrp.create_group('obs') 
    for grp_name in new_data['data'][demo]['obs']:
        dset = obsgrp.create_dataset(grp_name, data = new_data['data'][demo]['obs'][grp_name])
print('demo done', demo)
f.close()


demo done demo_1007


In [100]:
print(trial_hammer_basepath+'data_all.hdf5')
print(h5py.File(trial_hammer_basepath+'data_all.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_unsuccessful_only.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_successful_only.hdf5')['data'])

/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/alift_mugbeige_31_14_13_30/data_all.hdf5
<HDF5 group "/data" (1008 members)>
<HDF5 group "/data" (134 members)>
<HDF5 group "/data" (874 members)>


In [101]:
'''TEST THE NEW DATASET'''

data = h5py.File(trial_hammer_basepath + 'data_all.hdf5', 'r')
print(data['data'])
video_path = trial_hammer_basepath+'temp.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 50
demo = f'demo_{idx}'
print('shape', data['data'][demo]['obs']['agentview_image'].shape)
print('success', data['data'][demo]['success'][:])
print('ogpt', data['data'][demo]['og_pt'][0])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (84, 84) to (96, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


<HDF5 group "/data" (1008 members)>
shape (100, 84, 84, 3)
success [0.]
ogpt 50


[swscaler @ 0x579e280] Warning: data is not aligned! This can lead to a speed loss


# Testing other datasets

In [84]:
data=h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/needle2_seed6000/data_all.hdf5', 'r')
print(data['data'])
video_path = 'temp.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 60
demo = f'demo_{idx}'
if 'success' in data['data'][demo]:
    print('success',data['data'][demo]['success'][0])
print('shape', data['data'][demo]['obs']['agentview_image'].shape)
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)

maxd = 0
for b in  data['data']:
    maxd = max(data['data'][b]['obs']['agentview_image'].shape[0], maxd)
print('maxd', maxd)
data.close()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (84, 84) to (96, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


<HDF5 group "/data" (1008 members)>
success 1.0
shape (68, 84, 84, 3)


[swscaler @ 0x68dd1c0] Warning: data is not aligned! This can lead to a speed loss


maxd 100


In [50]:
h5py.File(trial_hammer_basepath + 'data_all.hdf5', 'r')['data']['demo_0'].keys()

<KeysViewHDF5 ['actions', 'obs', 'og_pt', 'rewards', 'states', 'success']>

# Combine Datasets

In [ ]:
temp = h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/robomimic/datasets/image_abs.hdf5')

In [ ]:
temp['data'].keys()

In [117]:
''' CREATE 1 DATASET '''
paths = ['/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/greencube2_seed6000/data_all.hdf5', 
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/redcube2_seed6000/data_all.hdf5',
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/hammer2_seed6000/data_all.hdf5',
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/mugbeige2_seed6000/data_all.hdf5',
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/mugred2_seed6000/data_all.hdf5',
         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/needle2_seed6000/data_all.hdf5']

# paths= ['/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/robomimic/datasets/image_abs.hdf5',
#         '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/needle2/data_successful_only.hdf5']
# Open HDF5 file and write in the data_dict structure and info
base_path = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/curateddata/combined2_seed6000/'
savepath = base_path+'data_all.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['env_args'] = og_redcube_data['data'].attrs['env_args']

count=0
for current_path in paths:
    if 'hammer' in current_path:
        current_object = 'hammer'
    elif 'needle' in current_path:
        current_object = 'needle'
    elif 'greencube' in current_path:
        current_object = 'greencube'
    elif 'mugbeige' in current_path:
        current_object = 'mugbeige'
    elif 'robomimic/datasets/image_abs' in current_path:
        current_object = 'redcube'
    current_dataset = h5py.File(current_path)
    for demo in current_dataset['data']:
        demogrp = datagrp.create_group('demo_'+str(count))
        objectsdset = demogrp.create_dataset('object', data = current_object)
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        if current_object == 'redcube':
            successdset = demogrp.create_dataset('success', data = [1.0])
        else:
            successdset = demogrp.create_dataset('success', data = current_dataset['data'][demo]['success'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        # statesdset = demogrp.create_dataset('og_pt', data = current_dataset['data'][demo]['og_pt'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        print('demo done', count)
        count += 1

    print('dataset done', current_path)

datagrp.attrs['total'] = count
f.close()

demo done 0
demo done 1
demo done 2
demo done 3
demo done 4
demo done 5
demo done 6
demo done 7
demo done 8
demo done 9
demo done 10
demo done 11
demo done 12
demo done 13
demo done 14
demo done 15
demo done 16
demo done 17
demo done 18
demo done 19
demo done 20
demo done 21
demo done 22
demo done 23
demo done 24
demo done 25
demo done 26
demo done 27
demo done 28
demo done 29
demo done 30
demo done 31
demo done 32
demo done 33
demo done 34
demo done 35
demo done 36
demo done 37
demo done 38
demo done 39
demo done 40
demo done 41
demo done 42
demo done 43
demo done 44
demo done 45
demo done 46
demo done 47
demo done 48
demo done 49
demo done 50
demo done 51
demo done 52
demo done 53
demo done 54
demo done 55
demo done 56
demo done 57
demo done 58
demo done 59
demo done 60
demo done 61
demo done 62
demo done 63
demo done 64
demo done 65
demo done 66
demo done 67
demo done 68
demo done 69
demo done 70
demo done 71
demo done 72
demo done 73
demo done 74
demo done 75
demo done 76
demo done

In [140]:
'''TEST THE NEW DATASET'''

data = h5py.File(savepath, 'r')
print(data['data'])
video_path = base_path+'temp1.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 6040
demo = f'demo_{idx}'
print(data['data'][demo]['object'])
print(data['data'][demo]['obs']['agentview_image'].shape)
print(data['data'][demo]['success'][:])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (84, 84) to (96, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


<HDF5 group "/data" (6048 members)>
<HDF5 dataset "object": shape (), type "|O">
(100, 84, 84, 3)
[0.]


[swscaler @ 0x569b080] Warning: data is not aligned! This can lead to a speed loss
